# 📊 Task 3 — Exploratory Data Analysis (EDA)
**DecodeLabs Data Analytics Internship | Batch 2026**  
**Intern:** Umm-e-Abiha

---

## 🎯 Objective
Systematically explore the cleaned dataset to identify statistical patterns, business metrics, outliers, and relationships between variables.

## 📋 Deliverables
- Descriptive statistics (mean, median, std, percentiles)
- Key business KPIs
- Outlier detection using the IQR method
- Correlation matrix and interpretation
- Revenue breakdown by product and order status
- Monthly revenue trend

## 🧠 Skills Applied
`descriptive statistics` · `IQR outlier detection` · `correlation analysis` · `groupby aggregation`

---

## ⚙️ Environment Setup, Data Load & Clean

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# ── Global Plot Style ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor'  : '#1a1a2e',
    'axes.edgecolor'  : '#00d4ff',
    'axes.labelcolor' : '#ffffff',
    'xtick.color'     : '#cccccc',
    'ytick.color'     : '#cccccc',
    'text.color'      : '#ffffff',
    'grid.color'      : '#2a2a4a',
    'grid.linestyle'  : '--',
    'grid.alpha'      : 0.4,
})

CYAN   = '#00d4ff'
ORANGE = '#ff6b35'
GREEN  = '#39ff14'
PURPLE = '#bf5af2'
YELLOW = '#ffd60a'
PINK   = '#ff2d78'
COLORS = [CYAN, ORANGE, GREEN, PURPLE, YELLOW, PINK, '#00ff88']

print("✅ All libraries imported successfully!")

df_raw = pd.read_excel('Dataset_for_Data_Analytics.xlsx')
df = df_raw.copy()
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')
df = df.drop_duplicates()
df['Date']      = pd.to_datetime(df['Date'])
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.strftime('%b')
for col in ['Product', 'OrderStatus', 'PaymentMethod', 'ReferralSource']:
    df[col] = df[col].str.strip().str.title()
print(f"Clean dataset ready: {df.shape[0]:,} rows × {df.shape[1]} columns")


## 3.1 Descriptive Statistics

In [ ]:
# ── Statistical Summary ───────────────────────────────────────────────────────
print("Descriptive Statistics — Numeric Columns:")
df[['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']].describe().round(2)


## 3.2 Key Business Metrics (KPIs)

In [ ]:
# ── Business KPIs ────────────────────────────────────────────────────────────
print("=" * 50)
print("   KEY BUSINESS METRICS")
print("=" * 50)
print(f"   💰 Average Order Value   : ${df['TotalPrice'].mean():>10.2f}")
print(f"   💰 Median Order Value    : ${df['TotalPrice'].median():>10.2f}")
print(f"   💰 Total Revenue         : ${df['TotalPrice'].sum():>12,.2f}")
print(f"   📈 Highest Single Order  : ${df['TotalPrice'].max():>10.2f}")
print(f"   📉 Lowest Single Order   : ${df['TotalPrice'].min():>10.2f}")
print(f"   📦 Total Orders          :  {len(df):>10,}")
print()
print(f"   🏆 Top Product (Revenue) : {df.groupby('Product')['TotalPrice'].sum().idxmax()}")
print(f"   💳 Top Payment Method    : {df['PaymentMethod'].value_counts().idxmax()}")
print(f"   📣 Top Referral Source   : {df['ReferralSource'].value_counts().idxmax()}")
print(f"   🏷️  Coupon Usage Rate     : {(df['CouponCode'] != 'NO_COUPON').mean()*100:.1f}%")


## 3.3 Outlier Detection — IQR Method

In [ ]:
# ── IQR-Based Outlier Detection ──────────────────────────────────────────────
print(f"{'Column':<15} {'Q1':>8} {'Q3':>8} {'IQR':>8} {'Lower Fence':>12} {'Upper Fence':>12} {'Outliers':>9}")
print("─" * 76)

for col in ['TotalPrice', 'UnitPrice', 'Quantity', 'ItemsInCart']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    n_out  = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"  {col:<13} {Q1:>8.1f} {Q3:>8.1f} {IQR:>8.1f} {lower:>12.1f} {upper:>12.1f} {n_out:>9}")


## 3.4 Correlation Matrix

In [ ]:
# ── Correlation Analysis ─────────────────────────────────────────────────────
corr = df[['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']].corr().round(3)
print("Correlation Matrix:")
print(corr.to_string())
print()
print("Interpretation:")
print(f"  UnitPrice  ↔ TotalPrice  : {corr.loc['TotalPrice','UnitPrice']:.3f}  → Strong positive correlation")
print(f"  Quantity   ↔ TotalPrice  : {corr.loc['TotalPrice','Quantity']:.3f}  → Moderate positive correlation")
print(f"  Quantity   ↔ ItemsInCart : {corr.loc['Quantity','ItemsInCart']:.3f}  → Moderate positive correlation")


## 3.5 Revenue Breakdown by Product

In [ ]:
# ── Revenue by Product ───────────────────────────────────────────────────────
rev_prod = df.groupby('Product')['TotalPrice'].agg(
    Total_Revenue='sum',
    Avg_Order_Value='mean',
    Order_Count='count'
).round(2).sort_values('Total_Revenue', ascending=False)

print("Revenue by Product:")
rev_prod


## 3.6 Revenue by Order Status

In [ ]:
# ── Revenue by Order Status ──────────────────────────────────────────────────
status_rev = df.groupby('OrderStatus')['TotalPrice'].agg(
    Total_Revenue='sum',
    Avg_Order_Value='mean',
    Order_Count='count'
).round(2).sort_values('Total_Revenue', ascending=False)

print("Revenue by Order Status:")
status_rev


## 3.7 Monthly Revenue Trend

In [ ]:
# ── Monthly Revenue ───────────────────────────────────────────────────────────
monthly_rev = df.groupby(['Year', 'Month'])['TotalPrice'].sum().reset_index()
monthly_rev['Period'] = (monthly_rev['Year'].astype(str) + '-' +
                          monthly_rev['Month'].astype(str).str.zfill(2))
monthly_rev = monthly_rev.sort_values(['Year', 'Month'])
print("Monthly Revenue (first 8 periods):")
print(monthly_rev.head(8).to_string(index=False))


## 3.8 EDA Visualisation Dashboard

In [ ]:
# ── Task 3 — 6-Panel EDA Dashboard ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('Task 3 — Exploratory Data Analysis Dashboard',
             color=CYAN, fontsize=16, fontweight='bold', y=1.01)

# 1. TotalPrice Distribution
axes[0,0].hist(df['TotalPrice'], bins=40, color=CYAN, edgecolor='#0d0d0d', alpha=0.85)
axes[0,0].axvline(df['TotalPrice'].mean(),   color=ORANGE, linestyle='--', lw=2,
                   label=f"Mean: ${df['TotalPrice'].mean():.0f}")
axes[0,0].axvline(df['TotalPrice'].median(), color=GREEN, linestyle='--', lw=2,
                   label=f"Median: ${df['TotalPrice'].median():.0f}")
axes[0,0].set_title('TotalPrice Distribution', color=CYAN, fontsize=12)
axes[0,0].set_xlabel('Total Price ($)'); axes[0,0].set_ylabel('Frequency')
axes[0,0].legend(fontsize=9)

# 2. Revenue by Product
rev_sorted = df.groupby('Product')['TotalPrice'].sum().sort_values()
bars = axes[0,1].barh(rev_sorted.index, rev_sorted.values,
                       color=COLORS[:len(rev_sorted)], edgecolor='#0d0d0d')
axes[0,1].set_title('Total Revenue by Product', color=CYAN, fontsize=12)
axes[0,1].set_xlabel('Revenue ($)')
for bar in bars:
    w = bar.get_width()
    axes[0,1].text(w + 200, bar.get_y() + bar.get_height()/2,
                   f'${w:,.0f}', va='center', fontsize=8, color='white')

# 3. Correlation Heatmap
corr_data = df[['Quantity','UnitPrice','ItemsInCart','TotalPrice']].corr()
im = axes[0,2].imshow(corr_data, cmap='coolwarm', vmin=-1, vmax=1)
axes[0,2].set_xticks(range(4)); axes[0,2].set_yticks(range(4))
axes[0,2].set_xticklabels(corr_data.columns, rotation=30, ha='right', fontsize=9)
axes[0,2].set_yticklabels(corr_data.columns, fontsize=9)
for i in range(4):
    for j in range(4):
        axes[0,2].text(j, i, f'{corr_data.iloc[i,j]:.2f}',
                       ha='center', va='center', fontsize=11, color='white', fontweight='bold')
axes[0,2].set_title('Correlation Heatmap', color=CYAN, fontsize=12)
plt.colorbar(im, ax=axes[0,2], shrink=0.8)

# 4. Boxplot
products = df['Product'].unique()
bp = axes[1,0].boxplot([df[df['Product']==p]['TotalPrice'].values for p in products],
                        patch_artist=True, medianprops=dict(color=ORANGE, linewidth=2))
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1,0].set_xticklabels(products, rotation=30, ha='right', fontsize=9)
axes[1,0].set_title('Price Distribution by Product', color=CYAN, fontsize=12)
axes[1,0].set_ylabel('Total Price ($)')

# 5. Order Status Count
status_c = df['OrderStatus'].value_counts()
axes[1,1].bar(status_c.index, status_c.values,
              color=COLORS[:len(status_c)], edgecolor='#0d0d0d', linewidth=1.5)
axes[1,1].set_title('Orders by Status', color=CYAN, fontsize=12)
axes[1,1].set_ylabel('Count'); axes[1,1].tick_params(axis='x', rotation=25)
for i, v in enumerate(status_c.values):
    axes[1,1].text(i, v + 2, str(v), ha='center', color='white', fontsize=10)

# 6. Referral Source Pie
ref_c = df['ReferralSource'].value_counts()
axes[1,2].pie(ref_c.values, labels=ref_c.index, autopct='%1.1f%%',
              colors=COLORS[:len(ref_c)],
              wedgeprops=dict(edgecolor='#0d0d0d', linewidth=1.5),
              textprops={'color': 'white', 'fontsize': 9})
axes[1,2].set_title('Referral Source Breakdown', color=CYAN, fontsize=12)

plt.tight_layout()
plt.savefig('task3_eda.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("✅ Chart saved → task3_eda.png")


## 💡 Task 3 — Key Findings

### Statistical Insights
| Metric | Value | Interpretation |
|--------|-------|----------------|
| Mean Order Value | ~$1,054 | Average spend per transaction |
| Median Order Value | ~$824 | 50th percentile of order values |
| Max Order | ~$3,456 | Highest single transaction |
| Min Order | ~$11 | Lowest single transaction |

> **Mean > Median** → Right-skewed distribution driven by a small number of high-value orders.

### Outlier Analysis (IQR)
- **TotalPrice:** 8 outliers — high-value orders (signal, not noise)
- **UnitPrice, Quantity, ItemsInCart:** 0 outliers — well-bounded distributions

### Correlation Insights
| Relationship | Coefficient | Strength |
|---|---|---|
| UnitPrice ↔ TotalPrice | ~0.72 | Strong positive |
| Quantity ↔ TotalPrice | ~0.62 | Moderate positive |
| Quantity ↔ ItemsInCart | ~0.65 | Moderate positive |

### Revenue Insights
- **Chair** generates the highest total revenue
- **Instagram** is the top customer acquisition channel
- Revenue is distributed relatively evenly across order statuses
